# Debruiteurs Deep

In [ ]:
from denoiser import pretrained #DEMUCS
from denoiser import demucs
import sys
import numpy as np
import matplotlib.pyplot as plt
import soundfile as sf
from IPython.display import Audio, display
from denoiser.dsp import convert_audio
import torch
import denoiser

sys.path.append("../utils")
import noise
from tools import listen

In [ ]:
#Import des fichiers sonores et verification de l'echantillonage
son1,fs1=sf.read('../../data/son1.wav')
print('Echantillonage 1 :',fs1)
son2,fs2=sf.read('../../data/son2.wav')
print('Echantillonage 2 :',fs2)
son3,fs3=sf.read('../../data/son3.wav')
print('Echantillonage 3 :',fs3)
son4,fs4=sf.read('../../data/son4.wav')
print('Echantillonage 4 :',fs4)

### DEMUCS denoiser

In [ ]:
isnr=1
son1_b,s1=noise.add_white_noise(son1,fs1,isnr)
listen(son1_b,44100)
son2_b,s2=noise.add_white_noise(son2,fs2,isnr)
listen(son2_b,44100)

In [ ]:
help(denoiser)

In [ ]:
help(pretrained)

In [ ]:
print(pretrained.dns64().sample_rate)
print(pretrained.dns64().chin)

In [ ]:
help(convert_audio)

In [ ]:
#Son vocal
son1_bt=convert_audio(torch.from_numpy(son1_b).float()[None, :],44100,16000,1)
model=pretrained.dns64()
son1_dt=model(son1_bt[None])
son1_d=son1_dt[0, 0].detach().cpu().numpy()
listen(son1_d,16000)

In [ ]:
#Son musical
son2_bt=convert_audio(torch.from_numpy(son2_b).float()[None, :],44100,16000,1)
model=pretrained.dns64()
son2_dt=model(son2_bt[None])
son2_d=son2_dt[0, 0].detach().cpu().numpy()
listen(son2_d,16000)

#### Conclusion DEMUCS
On remarque que le debruiteur DEMUCS filtre le son vocal correctement mais produit un résultat incohérent pour le son musical car il est entrainé sur de la parole et non de la musique.

## SGMSE+

In [ ]:
from sgmse.model import ScoreModel
from sgmse.util.other import pad_spec
from metrics import SNR

In [ ]:
device='cpu'

In [ ]:
def denoise_sgmse(sound,ckpt_path):
    model=ScoreModel.load_from_checkpoint(ckpt_path, map_location=device)
    model.eval()
    model.to(device)
    sound=sound.astype(np.float32)
    son_t=convert_audio(torch.from_numpy(sound).float()[None, :],44100,16000,1)
    Y=model._forward_transform(model._stft(son_t))
    Y=Y[None]
    Y=pad_spec(Y, mode="reflection")
    sampler = model.get_pc_sampler("reverse_diffusion","ald",Y.to(device),N=2,corrector_steps=1,snr=.5,)
    with torch.no_grad():
        sample, _ = sampler()
    son_dt=model.to_audio(sample.squeeze(), son_t.shape[-1])
    son_d=son_dt.detach().cpu().numpy()
    return son_d

In [ ]:
son4_d_sgmse=denoise_sgmse(son4,"../../Data/train_wsj0_2cta4cov_epoch=159.ckpt")

In [ ]:
listen(son4_d_sgmse,16000)

In [ ]:
son1_d_sgmse=denoise_sgmse(son1,"../../Data/train_wsj0_2cta4cov_epoch=159.ckpt")

In [ ]:
listen(son1_d_sgmse,16000)

In [ ]:
#son4
#Convertit le fichier audio de 44100 Hz a 16000 Hz
son4_t1=convert_audio(torch.from_numpy(son4).float()[None, :],44100,16000,1)[0].detach().cpu().numpy()
print("SGMSE+ SNR =",SNR(son4_t1,son4_d_sgmse),'dB')

In [ ]:
#son1
#Convertit le fichier audio de 44100 Hz a 16000 Hz
son1_t1=convert_audio(torch.from_numpy(son1).float()[None, :],44100,16000,1)[0].detach().cpu().numpy()
print("SGMSE+ SNR =",SNR(son1_t1,son1_d_sgmse),'dB')